# Skincare Review NLP: Sentiment Classification

This notebook demonstrates an end-to-end Natural Language Processing (NLP) pipeline to classify the sentiment of customer reviews for skincare products into **positive**, **neutral**, or **negative** classes.

### Workflow Overview:
1. **Environment Setup & Imports**: Load libraries needed for data handling, plotting, and modeling.
2. **Data Generation**: Create a synthetic dataset simulating realistic skincare reviews to make the notebook self-contained.
3. **Exploratory Data Analysis (EDA)**: Examine the target distributions and review word counts.
4. **Text Preprocessing**: Clean the raw text data (lowercasing, punctuation removal, stopword filtering, and lemmatization).
5. **Feature Extraction**: Convert cleaned text into numerical vectors using Term Frequency-Inverse Document Frequency (TF-IDF).
6. **Model Training & Evaluation**: Train a Logistic Regression model and evaluate its performance.
7. **Inference Pipeline**: Package the preprocessing and classification steps into a prediction function for unseen reviews.

In [ ]:
# 1. Environment Setup & Imports
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

print("Imports completed successfully.")

## 2. Dataset Generation

We simulate customer feedback to create a robust sample dataset. This ensures that the notebook can be executed immediately without external files.

In [ ]:
# Define simulated skincare reviews
raw_reviews = [
    {"review": "I love this moisturizer! It keeps my skin hydrated all day and does not break me out.", "sentiment": "positive"},
    {"review": "Absolutely terrible. This serum gave me a huge rash and my skin is extremely dry now.", "sentiment": "negative"},
    {"review": "It is an okay cleanser, nothing special. It cleans my face but feels a bit stripping.", "sentiment": "neutral"},
    {"review": "Best sunscreen ever! No white cast, absorbs quickly, and works great under makeup.", "sentiment": "positive"},
    {"review": "Highly disappointed. After using this for a week, my acne got worse. Avoid!", "sentiment": "negative"},
    {"review": "The product is average. It does the job, but the fragrance is too strong.", "sentiment": "neutral"},
    {"review": "My skin has never looked better. This exfoliator cleared up my blackheads completely.", "sentiment": "positive"},
    {"review": "Waste of money. It did absolutely nothing for my dark spots.", "sentiment": "negative"},
    {"review": "It is a decent moisturizer for the price, but there are better options out there.", "sentiment": "neutral"},
    {"review": "Amazing! This night cream made my skin so soft and glowing by morning.", "sentiment": "positive"},
    {"review": "This facial oil is too greasy. It felt heavy on my skin and clogged my pores.", "sentiment": "negative"},
    {"review": "The toner is fine, nothing remarkable. Just feels like water on my face.", "sentiment": "neutral"},
    {"review": "Perfect for sensitive skin. No redness or irritation whatsoever.", "sentiment": "positive"},
    {"review": "Smells horrible and burned my face. I had to wash it off immediately.", "sentiment": "negative"},
    {"review": "It is alright, but I have not noticed any major changes in my skin texture.", "sentiment": "neutral"},
    {"review": "I highly recommend this product. It helped fade my hyperpigmentation significantly.", "sentiment": "positive"},
    {"review": "It dried out my skin completely. Now my face is peeling and red.", "sentiment": "negative"},
    {"review": "Not bad, but not great either. Just a basic hydrating serum.", "sentiment": "neutral"},
    {"review": "This is my holy grail! My dry skin finally feels balanced and nourished.", "sentiment": "positive"},
    {"review": "Terrible customer service and the pump arrived broken. The cream itself is mediocre.", "sentiment": "negative"}
]

# Replicate dataset to scale the dimensions for training
dataset = raw_reviews * 15
df = pd.DataFrame(dataset)

# Shuffle and reset index
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

## 3. Exploratory Data Analysis (EDA)

Understanding text data attributes before applying model preprocessing.

In [ ]:
# View class counts
sentiment_counts = df['sentiment'].value_counts()
print("Sentiment Distribution:")
print(sentiment_counts)

# Calculate review word count
df['word_count'] = df['review'].apply(lambda x: len(x.split()))

# Plotting
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Class distribution plot
sns.countplot(data=df, x='sentiment', ax=axes[0], palette='pastel')
axes[0].set_title('Review Sentiment Distribution')
axes[0].set_xlabel('Sentiment')
axes[0].set_ylabel('Count')

# Word count distribution by sentiment
sns.boxplot(data=df, x='sentiment', y='word_count', ax=axes[1], palette='pastel')
axes[1].set_title('Review Length by Sentiment')
axes[1].set_xlabel('Sentiment')
axes[1].set_ylabel('Word Count')

plt.tight_layout()
plt.show()

## 4. Text Preprocessing

Standard cleaning practices to map input features into a clean vocabulary:
- Lowercase conversion.
- Punctuation and non-alphabetical character extraction.
- Stopword elimination.
- Word lemmatization (reducing words to their dictionary forms).

In [ ]:
# Download resource dependencies
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    # Lowercase conversion
    text = text.lower()
    # Keep only letters and spaces
    text = re.sub('[^a-zA-Z ]', '', text)
    # Split tokens
    words = text.split()
    # Filter stopwords and apply lemmatization
    cleaned_words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    return " ".join(cleaned_words)

# Process dataset reviews
df['cleaned_review'] = df['review'].apply(preprocess_text)

# Display a transformation sample
print("Raw Review:")
print(df['review'].iloc[0])
print("\nPreprocessed Review:")
print(df['cleaned_review'].iloc[0])

## 5. Feature Extraction & Train-Test Split

We split the dataset into stratified validation splits (75% train, 25% test) to preserve category balance, then convert clean texts into TF-IDF features containing unigrams and bigrams.

In [ ]:
# Define features and targets
X = df['cleaned_review']
y = df['sentiment']

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.25, 
    random_state=42, 
    stratify=y
)

# Initialize TF-IDF Vectorizer
tfidf = TfidfVectorizer(max_features=1000, ngram_range=(1, 2))

# Fit on training split and transform test split
X_train_vec = tfidf.fit_transform(X_train)
X_test_vec = tfidf.transform(X_test)

print(f"Training set shape: {X_train_vec.shape}")
print(f"Testing set shape: {X_test_vec.shape}")

## 6. Model Training & Evaluation

We construct a Logistic Regression model adjusting class weights to manage category counts. Model predictions are validated through performance metrics and graphical confusion matrix overlays.

In [ ]:
# Fit Logistic Regression classifier
clf = LogisticRegression(class_weight='balanced', random_state=42)
clf.fit(X_train_vec, y_train)

# Classify validation split
y_pred = clf.predict(X_test_vec)

# Output evaluations
print("Validation Accuracy Score:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Confusion Matrix mapping
labels = ['positive', 'neutral', 'negative']
cm = confusion_matrix(y_test, y_pred, labels=labels)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=labels, yticklabels=labels, cmap='Blues')
plt.title('Confusion Matrix Representation')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

## 7. Inference Pipeline

We package the processing functions and modeling artifacts into a reusable inference helper to evaluate unseen reviews.

In [ ]:
def predict_sentiment(review):
    # Run cleaner
    cleaned = preprocess_text(review)
    # Vectorize inputs
    vectorized = tfidf.transform([cleaned])
    # Predict outcomes
    prediction = clf.predict(vectorized)[0]
    probs = clf.predict_proba(vectorized)[0]
    
    confidence_mapping = {clf.classes_[i]: f"{probs[i]:.2%}" for i in range(len(clf.classes_))}
    
    return {
        "review_raw": review,
        "predicted_sentiment": prediction,
        "confidence_scores": confidence_mapping
    }

# Trial evaluations
new_reviews = [
    "This cream left my face slightly sticky, but the hydration level is adequate overall.",
    "My skin immediately developed red spots and felt itchy. Would not buy again."
]

for r in new_reviews:
    res = predict_sentiment(r)
    print(f"Review: '{res['review_raw']}'")
    print(f"Prediction: {res['predicted_sentiment']}")
    print(f"Confidences: {res['confidence_scores']}\n")